# Transformer-Based Personality Trait Recognition
### A Beginner-Friendly Guide to Running This on Google Colab

---

## What is this project?

This project trains an AI model to predict **Big Five personality traits** (Openness, Conscientiousness, Extraversion, Agreeableness, Neuroticism) from written text (essays). It uses a transformer model called **ELECTRA**, developed by Google.

Think of it like teaching a model to read someone's writing and guess whether they score high or low on each personality dimension.


## Why Google Colab?

Training deep learning models requires a **GPU** (a specialised processor that handles heavy mathematical computations much faster than a regular CPU). The original code was written to run on a machine with an NVIDIA GPU.

If you are on a **Mac, a laptop without a dedicated GPU, or any machine without NVIDIA hardware**, Google Colab gives you free access to a GPU (the T4) in the cloud — no setup required on your own machine.


## Why Google Drive?

Google Colab sessions are **temporary**. When your session disconnects or times out, everything stored in the Colab environment (`/content/`) is permanently deleted — including any trained models that took hours to produce.

By connecting Google Drive and working directly from it, all files (trained models, results, logs) are saved to your personal Drive in real time. Even if Colab disconnects mid-training, your progress is safe.


## Before you start

1. Open this notebook in **Google Colab** (upload it or open from Drive)
2. Set the runtime to use a GPU: **Runtime → Change runtime type → T4 GPU**
3. Run the cells **in order**, top to bottom
4. You will need a Google account for Drive access

---
## ⚠️ Important Notes Before You Start

Based on real experience running this notebook on Google Colab, please read these before proceeding:

1. **Save everything to Google Drive.** Colab sessions are temporary — when a session ends, all files in the Colab environment are permanently deleted. This notebook saves all files directly to your Drive (Step 1), which is essential. Do not skip that step or change the paths.

2. **You need ~22 GB of free Google Drive space.** Training all five models produces roughly 22 GB of checkpoint files. Check your available storage at [drive.google.com](https://drive.google.com) before starting. If needed, free up space or purchase additional storage.

3. **You cannot train all 5 traits in a single free-tier session.** The free T4 GPU has a session time limit. Expect to need at least 2 separate sessions to train all five traits. If you want to complete everything in one go, consider upgrading to **Colab Pro**.

---
## Step 1 · Connect Google Drive and Set Up the Project

This is the most important step. We mount your Google Drive so that the entire project — code, data, and trained models — lives there permanently.

When you run this cell, Google will ask you to authorise access to your Drive. Follow the link, sign in, and paste the code back. This only needs to be done once per session.

In [ ]:
from google.colab import drive
import os

# Connect your Google Drive to this Colab session
drive.mount('/content/drive')

# This is the folder we will create inside your Drive.
# Everything — code, data, trained models — will live here.
PROJECT_DIR = '/content/drive/MyDrive/single-trait-electra'

REPO_URL = 'https://github.com/hsisaberi/single-trait-electra'

if not os.path.exists(PROJECT_DIR):
    # First time: clone the repository directly into your Drive
    !git clone {REPO_URL} {PROJECT_DIR}
    print('Project cloned to your Google Drive.')
else:
    # Project already exists in Drive: just pull any updates
    !git -C {PROJECT_DIR} pull
    print('Project already exists in Drive. Pulled latest changes.')

# Set this folder as our working directory for the rest of the notebook
%cd {PROJECT_DIR}
print('\nWorking directory:', os.getcwd())

---
## Step 2 · Install Dependencies

The project requires several Python libraries (transformers, PyTorch, scikit-learn, etc.).

**Why two separate install commands?**  
Colab comes with PyTorch and torchvision pre-installed, but they can be mismatched versions if we blindly reinstall everything from the requirements file. We install `torch` and `torchvision` together first as a guaranteed compatible pair, then install the rest.

⚠️ **After running both cells below, you must restart the runtime before continuing.**  
This is because Python caches the old library versions in memory — a restart forces it to load the freshly installed ones.

In [ ]:
# Step 2a — Install PyTorch and torchvision as a matched compatible pair
!pip install torch torchvision -q

In [ ]:
# Step 2b — Install all remaining dependencies from the project requirements
# (torch/CUDA/triton packages are skipped since we handled them above)
!grep -vE '^torch|^torchvision|^nvidia-|^triton' requirements.txt \
    | pip install -r /dev/stdin -q

### 🔁 Restart the runtime now

Go to **Runtime → Restart session**, then continue from Step 3.  
Do **not** re-run Steps 1 and 2 after restarting — they are already done.

---
## Step 3 · Resume After Restart

After restarting, the working directory resets. This cell reconnects Drive and returns to the project folder.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/single-trait-electra'
%cd {PROJECT_DIR}
print('Working directory:', os.getcwd())

---
## Step 4 · Verify the GPU

This confirms that Colab has assigned a GPU to your session. If `CUDA available` shows `False`, go to **Runtime → Change runtime type** and select T4 GPU.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

---
## Step 5 · Patch GPU Device References

The original scripts were written for a machine with **multiple GPUs** and reference `cuda:4` or `cuda:5`. Colab provides a single GPU referenced as `cuda:0`.

This cell automatically rewrites those references in all scripts so they work on Colab. You only need to run this once per project setup.

In [ ]:
import re, pathlib

FILES_TO_PATCH = [
    'train.py', 'eval.py', 'kfold_train.py',
    'infer.py', 'grid_search/grid_search.py'
]

for fname in FILES_TO_PATCH:
    p = pathlib.Path(fname)
    if not p.exists():
        print(f'Not found (skipping): {fname}')
        continue
    original = p.read_text()
    patched  = re.sub(r'cuda:\d+', 'cuda:0', original)
    if patched != original:
        p.write_text(patched)
        print(f'Patched: {fname}')
    else:
        print(f'Already correct: {fname}')

---
## Step 6 · Download Language Resources (NLTK)

The data augmentation part of this project uses **NLTK** (Natural Language Toolkit), a library for processing text. It needs to download some language data files (like a dictionary of synonyms) before it can work.

These will be saved inside the project folder on your Drive.

In [ ]:
!python nltk_res/nltk_res_donwload.py

---
## Step 7 · Split the Dataset

The project uses the **Pennebaker & King Essays dataset** — 2,467 essays each labelled with Big Five personality scores.

Before training, we divide the data into three parts:
- **Train** (~70%) — the data the model learns from
- **Validation** (~15%) — used during training to check if the model is improving or overfitting
- **Test** (~15%) — held back completely until the end to give an honest measure of performance

This split is done once and the results are saved to `dataset/split/` on your Drive.

In [ ]:
!python data_splitter.py

In [ ]:
# Verify the split was created correctly
import pandas as pd

for split in ['train', 'validation', 'test']:
    df = pd.read_csv(f'dataset/split/{split}.csv')
    print(f'{split:12s}: {len(df):>5} rows')

---
## Step 8 · Train a Single-Trait Model

### What is training?

Training is the process of teaching the model to make predictions. We start with ELECTRA — a model pre-trained by Google on a massive amount of text (so it already understands language). We then **fine-tune** it on our essay dataset, teaching it to predict personality traits specifically.

During training, the model makes predictions, compares them to the correct answers, measures the error (called **loss**), and adjusts its internal parameters to do better next time. This repeats for **10 epochs** (10 full passes through the training data).

### What to expect

- Each epoch takes roughly **3–4 minutes** on a T4 GPU — so expect around 30–40 minutes per trait
- You will see `Train Loss` decrease over epochs — this means the model is learning
- `Validation Accuracy` is the more meaningful number — it shows how well the model generalises to data it hasn't seen
- A checkpoint (`.pt` file) is saved to your Drive after every epoch

### One model per trait

This project trains **five separate models**, one per personality trait. Change `TRAIT_NAME` below and run this section for each trait.

In [ ]:
# ── Set the trait you want to train ──────────────────────────────────────────
TRAIT_NAME = 'Neuroticism'  # Options: Openness | Conscientiousness | Extroversion | Agreeableness | Neuroticism
# ─────────────────────────────────────────────────────────────────────────────

import re, pathlib

src = pathlib.Path('train.py').read_text()
src = re.sub(r'self\.trait_name\s*=\s*"[^"]+"', f'self.trait_name = "{TRAIT_NAME}"', src)
pathlib.Path('train.py').write_text(src)
print(f'Training target set to: {TRAIT_NAME}')
print(f'Checkpoints will be saved to: model/{TRAIT_NAME}/ (on your Drive)')

In [ ]:
!python train.py

> **To train all five traits**, change `TRAIT_NAME` in the cell above and re-run both cells for each of:  
> `Openness`, `Conscientiousness`, `Extroversion`, `Agreeableness`, `Neuroticism`
>
> All checkpoints are automatically saved to your Google Drive as training progresses.

---
## Step 9 · K-Fold Cross-Validation Training

### What is K-Fold Cross-Validation?

In Step 8, the model was trained on one fixed split of the data. The risk is that the results might be lucky (or unlucky) depending on which particular essays ended up in the training set.

K-Fold cross-validation addresses this by training the model **K times**, each time using a different portion of the data as the validation set. The final performance is the average across all K runs, giving a much more reliable and honest measure of how good the model actually is.

### What to expect

- This runs **10 folds × 10 epochs = 100 training runs** — significantly longer than Step 8
- Expect several hours of runtime
- Results and model checkpoints are saved to your Drive throughout, so a disconnection won't lose your progress
- At the end you will see average accuracy, F1, precision, and recall across all folds

In [ ]:
# ── Set the trait for K-Fold training ────────────────────────────────────────
KFOLD_TRAIT = 'Openness'  # change as needed
# ─────────────────────────────────────────────────────────────────────────────

import re, pathlib

src = pathlib.Path('kfold_train.py').read_text()
src = re.sub(r'self\.trait_name\s*=\s*"[^"]+"', f'self.trait_name = "{KFOLD_TRAIT}"', src)
pathlib.Path('kfold_train.py').write_text(src)
print(f'K-Fold training target set to: {KFOLD_TRAIT}')

In [ ]:
!python kfold_train.py

---
## Step 10 · Grid Search (Hyperparameter Optimisation)

### What is a grid search?

A model's behaviour is controlled by **hyperparameters** — settings like learning rate (how fast the model updates itself), batch size (how many essays it processes at once), and others. These are not learned during training; they are chosen beforehand.

A grid search tries every combination of hyperparameter values from a predefined list and evaluates which combination produces the best validation performance. Think of it like systematically taste-testing every possible recipe variation to find the best one.

The best configurations found by the authors are already saved in `grid_search/best_configs.csv`. Run this only if you want to redo the search yourself.

### What to expect

- This is the most time-consuming step — it runs many combinations of hyperparameters
- Results are saved incrementally to `grid_search/grid_search_results/` on your Drive

In [ ]:
!python -m grid_search.grid_search

---
## Step 11 · Evaluate a Trained Model

### What is evaluation?

After training, we run the model on the **test set** — data it has never seen before — to get an honest picture of its real-world performance. This produces metrics like:

- **Accuracy** — what percentage of predictions were correct
- **Precision** — of the essays predicted as "high", how many actually were
- **Recall** — of all the actual "high" essays, how many did the model catch
- **F1 Score** — a single number balancing precision and recall
- **ROC-AUC / PR-AUC** — curves that show model performance across all possible thresholds

### What to expect

The cell below first promotes your chosen epoch checkpoint to `best.pt` (the file `eval.py` looks for), then runs the evaluation. Results and plots are saved to `result/` on your Drive.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
EVAL_TRAIT  = 'Neuroticism'  # trait to evaluate # Options: Openness | Conscientiousness | Extroversion | Agreeableness | Neuroticism
BEST_EPOCH  = 10          # which epoch checkpoint to use as the final model
# ─────────────────────────────────────────────────────────────────────────────

import os, shutil

src = f'model/{EVAL_TRAIT}/model_epoch_{BEST_EPOCH}.pt'
dst = f'model/{EVAL_TRAIT}/best.pt'

if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Checkpoint promoted: {src} -> {dst}')
else:
    print(f'Checkpoint not found: {src}')
    print('Run Step 8 first, or adjust BEST_EPOCH to a completed epoch number.')

In [ ]:
import re, pathlib

src = pathlib.Path('eval.py').read_text()
src = re.sub(r'trait_name\s*=\s*"[^"]+"', f'trait_name = "{EVAL_TRAIT}"', src)
pathlib.Path('eval.py').write_text(src)
print(f'Evaluating: {EVAL_TRAIT}')

In [ ]:
!python eval.py

---
## Step 12 · Run Inference on New Text

### What is inference?

Inference is using the trained model to make predictions on brand new text — text the model has never seen and that has no labels. This is the end goal: given a piece of writing, predict the author's personality trait scores.

This step loads all five trained models and runs them on any text you provide. Each model independently predicts **High** or **Low** for its respective trait, along with a confidence score.

> **Prerequisite:** All five `model/<trait>/best.pt` files must exist (run Step 8 and the promote cell in Step 11 for each trait first).

In [ ]:
import os, torch
from transformers import ElectraTokenizerFast, ElectraForSequenceClassification

DEVICE     = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'google/electra-base-discriminator'

TRAIT_PATHS = {
    'Openness':          'model/Openness/best.pt',
    'Conscientiousness': 'model/Conscientiousness/best.pt',
    'Extroversion':      'model/Extroversion/best.pt',
    'Agreeableness':     'model/Agreeableness/best.pt',
    'Neuroticism':       'model/Neuroticism/best.pt',
}

tokenizer = ElectraTokenizerFast.from_pretrained(MODEL_NAME)

models = {}
for trait, path in TRAIT_PATHS.items():
    if not os.path.exists(path):
        print(f'[SKIP] {trait}: no checkpoint found at {path}')
        continue
    m = ElectraForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.to(DEVICE).eval()
    models[trait] = m
    print(f'Loaded: {trait}')

In [ ]:
# ── Enter any text you want to analyse ───────────────────────────────────────
TEXT = 'I am a very open, conscientious, extroverted, agreeable, and a neurotic person'
# ─────────────────────────────────────────────────────────────────────────────

encoding = tokenizer(
    TEXT, truncation=True, max_length=128,
    padding='max_length', return_tensors='pt'
).to(DEVICE)

print(f'\nText analysed:\n"{TEXT}"\n')
print(f'{"Personality Trait":<20} {"Prediction":>12} {"Confidence":>12}')
print('-' * 46)

for trait, model in models.items():
    with torch.no_grad():
        logits = model(**encoding).logits
        probs  = torch.softmax(logits, dim=1)[0]
        label  = torch.argmax(probs).item()
        conf   = probs[label].item()
    prediction = 'High' if label == 1 else 'Low'
    print(f'{trait:<20} {prediction:>12} {conf:>11.1%}')

---
## Step 13 · Data Augmentation  *(optional)*

### What is data augmentation?

The original dataset has 2,467 essays. More data generally means a better model, but collecting new labelled essays is expensive and time-consuming.

Data augmentation is a technique to **artificially expand the dataset** by creating modified versions of existing samples. This project uses two approaches:

1. **Synonym Replacement** — replaces some words with synonyms (e.g. "happy" → "joyful") to create a new but similar essay
2. **Contextual Augmentation** — uses a large language model (Gemma) to rewrite sentences while preserving meaning

The augmented datasets are already included in the repository (`dataset/augmentation/`), so **you do not need to run this** unless you want to regenerate them from scratch.

In [ ]:
# Synonym replacement augmentation — safe to run, uses WordNet locally
!python augment.py

In [ ]:
# Contextual augmentation — requires access to the Gemma model / API
# Only uncomment and run if you have the required access

# from augment import DataAugmentor
# DataAugmentor(aug_type='contextual').run()